# 🎵 TuneWave Music Player — Milestone 3: Song Search & Artist Recommendations
### Core Data Structures: Binary Search Tree (BST) + Undirected Graph (Adjacency List)

---

## 🎯 Milestone Goal & Problem Statement
Users need fast, responsive search to locate tracks by title, and an intelligent recommendation system that suggests new, musically adjacent artists without recommending artists the user is already familiar with.

### 📜 Formal Rules & Requirements:
1. **BST Title Indexing:** Songs are indexed in a **Binary Search Tree (BST)** keyed by `title`.
2. **Subtree Pruning Prefix Search:** `prefixSearch(prefix)` returns matching titles in **alphabetical order** and **skips subtrees that cannot match**.
3. **Undirected Artist Network:** Artists form an **undirected graph** of mutual related links.
4. **2-Hop Recommendations:** `recommendArtists(artist)` returns artists **exactly two links away** who are **not already directly related**, in **alphabetical order**.
5. **Required Operations:**
   - `addSong(title, songId)`
   - `findSong(title)`
   - `prefixSearch(prefix)`
   - `addRelation(artistA, artistB)`
   - `recommendArtists(artist)`


## 🏗️ Architecture & Theoretical Design

### 1. Binary Search Tree (BST) with Branch Pruning
```
                         [ "Hotel California" ]
                                /       \
                               /         \
               [ "Bohemian Rhapsody" ]   [ "Stairway to Heaven" ]
                       /     \                       \
                      /       \                       \
             [ "Blinding" ]  [ "Circles" ]          [ "Starboy" ]
                                                         \
                                                          \
                                                        [ "Stay" ]
```
- **Subtree Pruning Invariant:** For a prefix $P$ of length $L$:
  - If $P < 	ext{node.title}[:L]$, all nodes in the right subtree are strictly greater than $P \implies$ **Skip Right Subtree**.
  - If $P > 	ext{node.title}[:L]$, all nodes in the left subtree are strictly smaller than $P \implies$ **Skip Left Subtree**.
  - In-order traversal naturally yields results in **alphabetical order** without any sorting step!

---

### 2. Undirected Artist Relationship Graph
```
           [Drake] <--------------------> [The Weeknd] <--------------------> [Daft Punk]
              ^                                  |                                   |
              | (direct link)                    | (direct link)                     |
              v                                  v                                   v
        [Travis Scott]                     [Ariana Grande]                   [Pharrell Williams]
```
- **2-Hop Recommendation Invariant:**
  - $N_1 = 	ext{Direct Neighbors of } A$ (1 link away)
  - $N_2 = 	ext{Neighbors of all } u \in N_1$ (2 links away)
  - Filter: $v \in N_2$ such that $v 
eq A$ and $v 
otin N_1$ (suppresses triangle shortcuts and self-loops).
  - Return in strict **alphabetical order**.


In [1]:
from collections import defaultdict


class BSTNode:
    """Node in the Song Search Binary Search Tree."""
    def __init__(self, title: str, song_id: str):
        self.title = str(title)
        self.song_id = str(song_id)
        self.left = None
        self.right = None

    def __repr__(self):
        return f"BSTNode('{self.title}', '{self.song_id}')"


class SongSearchBST:
    """
    Binary Search Tree storing songs keyed by title.
    Enables O(log N) search and O(h + K) pruned prefix search.
    """
    def __init__(self):
        self.root = None
        self._size = 0

    def addSong(self, title: str, song_id: str):
        """Inserts song into BST keyed by title."""
        self.root = self._insert(self.root, str(title), str(song_id))
        self._size += 1

    def _insert(self, node: BSTNode, title: str, song_id: str) -> BSTNode:
        if node is None:
            return BSTNode(title, song_id)

        if title < node.title:
            node.left = self._insert(node.left, title, song_id)
        elif title > node.title:
            node.right = self._insert(node.right, title, song_id)
        else:
            node.song_id = song_id  # Update existing title ID
        return node

    def findSong(self, title: str) -> str:
        """Searches for exact title in BST. Returns songId or None."""
        curr = self.root
        target = str(title)
        while curr:
            if target == curr.title:
                return curr.song_id
            elif target < curr.title:
                curr = curr.left
            else:
                curr = curr.right
        return None

    def prefixSearch(self, prefix: str) -> list:
        """
        Returns matching titles in alphabetical order and skips subtrees that cannot match.
        """
        results = []
        prefix_str = str(prefix)
        if not prefix_str:
            return self.in_order_titles()

        self._prefix_search(self.root, prefix_str, results)
        return results

    def _prefix_search(self, node: BSTNode, prefix: str, results: list):
        if node is None:
            return

        L = len(prefix)
        node_prefix = node.title[:L]

        # 1. Prune Left Subtree if left keys cannot match
        if prefix <= node_prefix:
            self._prefix_search(node.left, prefix, results)

        # 2. Check current node
        if node.title.startswith(prefix):
            results.append(node.title)

        # 3. Prune Right Subtree if right keys cannot match
        if prefix >= node_prefix:
            self._prefix_search(node.right, prefix, results)

    def in_order_titles(self) -> list:
        """Returns all titles in in-order alphabetical sequence."""
        titles = []
        def _inorder(node):
            if node:
                _inorder(node.left)
                titles.append(node.title)
                _inorder(node.right)
        _inorder(self.root)
        return titles

    def __len__(self):
        return self._size


class ArtistGraph:
    """
    Undirected Graph of related artist links.
    """
    def __init__(self):
        self.adj = defaultdict(set)

    def addRelation(self, artistA: str, artistB: str):
        """Adds undirected symmetric link between artistA and artistB."""
        a, b = str(artistA), str(artistB)
        if a == b:
            return  # Ignore self-relationships
        self.adj[a].add(b)
        self.adj[b].add(a)

    def getDirectRelations(self, artist: str) -> set:
        """Returns 1-hop directly related artists."""
        return set(self.adj.get(str(artist), set()))

    def recommendArtists(self, artist: str) -> list:
        """
        Returns artists exactly two links away who are not already directly related,
        sorted in alphabetical order.
        """
        target = str(artist)
        if target not in self.adj:
            return []

        direct_relations = self.adj[target]
        candidates = set()

        for direct_neighbor in direct_relations:
            for second_degree in self.adj[direct_neighbor]:
                # Must be exactly 2 links away:
                # 1. Cannot be the original artist
                # 2. Cannot already be directly connected (1-hop)
                if second_degree != target and second_degree not in direct_relations:
                    candidates.add(second_degree)

        return sorted(list(candidates))


class TuneWaveDiscovery:
    """
    Unified Discovery Engine for Milestone 3.
    """
    def __init__(self):
        self.search_index = SongSearchBST()
        self.artist_network = ArtistGraph()

    def addSong(self, title: str, songId: str):
        self.search_index.addSong(title, songId)

    def findSong(self, title: str) -> str:
        return self.search_index.findSong(title)

    def prefixSearch(self, prefix: str) -> list:
        return self.search_index.prefixSearch(prefix)

    def addRelation(self, artistA: str, artistB: str):
        self.artist_network.addRelation(artistA, artistB)

    def recommendArtists(self, artist: str) -> list:
        return self.artist_network.recommendArtists(artist)


### 🧪 Interactive Demo 1: Indexing & Exact Search via BST
Adding songs into the BST index and verifying $O(h)$ exact lookups.


In [2]:
discovery = TuneWaveDiscovery()

# Index catalog of popular songs
songs_data = [
    ("Hotel California", "S101"),
    ("Bohemian Rhapsody", "S102"),
    ("Stairway to Heaven", "S103"),
    ("Blinding Lights", "S104"),
    ("Circles", "S105"),
    ("Starboy", "S106"),
    ("Stay", "S107"),
    ("Bad Guy", "S108"),
    ("Save Your Tears", "S109")
]

for title, s_id in songs_data:
    discovery.addSong(title, s_id)

print(f"Indexed {len(discovery.search_index)} songs into BST.")
print("\n-- Exact Search Tests --")
print("Find 'Starboy':", discovery.findSong("Starboy"))
print("Find 'Hotel California':", discovery.findSong("Hotel California"))
print("Find 'Non-Existent Track':", discovery.findSong("Non-Existent Track"))


Indexed 9 songs into BST.

-- Exact Search Tests --
Find 'Starboy': S106
Find 'Hotel California': S101
Find 'Non-Existent Track': None


### 🔍 Interactive Demo 2: Prefix Search with Subtree Pruning
Verifying that prefix search returns results in alphabetical order and skips subtrees that cannot match.


In [3]:
# Search for tracks starting with 'Sta'
sta_matches = discovery.prefixSearch("Sta")
print("Prefix query 'Sta' results (alphabetical):")
for match in sta_matches:
    print(f"  • {match} (ID: {discovery.findSong(match)})")

# Search for tracks starting with 'B'
b_matches = discovery.prefixSearch("B")
print("\nPrefix query 'B' results (alphabetical):")
for match in b_matches:
    print(f"  • {match} (ID: {discovery.findSong(match)})")

# Demonstrate Subtree Pruning in Action
visited_nodes = []
def instrumented_search(node, prefix):
    if not node:
        return
    visited_nodes.append(node.title)
    L = len(prefix)
    node_pref = node.title[:L]
    if prefix <= node_pref:
        instrumented_search(node.left, prefix)
    if prefix >= node_pref:
        instrumented_search(node.right, prefix)

instrumented_search(discovery.search_index.root, "Hotel")
total_nodes = len(discovery.search_index)
print(f"\nPruning Verification: Searching for 'Hotel' visited {len(visited_nodes)} nodes out of {total_nodes}!")
print("Visited path:", " -> ".join(visited_nodes))


Prefix query 'Sta' results (alphabetical):
  • Stairway to Heaven (ID: S103)
  • Starboy (ID: S106)
  • Stay (ID: S107)

Prefix query 'B' results (alphabetical):
  • Bad Guy (ID: S108)
  • Blinding Lights (ID: S104)
  • Bohemian Rhapsody (ID: S102)

Pruning Verification: Searching for 'Hotel' visited 5 nodes out of 9!
Visited path: Hotel California -> Bohemian Rhapsody -> Circles -> Stairway to Heaven -> Save Your Tears


### 🌐 Interactive Demo 3: Artist Recommendations (2 Links Away)
Building an undirected artist network and querying recommendations while verifying triangle filtering.


In [4]:
# Define mutual musical relations
relations = [
    ("The Weeknd", "Daft Punk"),
    ("The Weeknd", "Travis Scott"),
    ("The Weeknd", "Ariana Grande"),
    ("Daft Punk", "Pharrell Williams"),
    ("Daft Punk", "Kanye West"),
    ("Travis Scott", "Drake"),
    ("Travis Scott", "Kendrick Lamar"),
    ("The Weeknd", "Drake")  # Direct link! Drake is already directly related to The Weeknd
]

for a1, a2 in relations:
    discovery.addRelation(a1, a2)

print("Direct relationships for 'The Weeknd':")
print(discovery.artist_network.getDirectRelations("The Weeknd"))

# Request recommendations for 'The Weeknd'
weeknd_recs = discovery.recommendArtists("The Weeknd")
print("\nRecommendations for 'The Weeknd' (Exactly 2 links away, not already directly related):")
print(weeknd_recs)

print("\nNotice:")
print("• 'Pharrell Williams' and 'Kanye West' are recommended via Daft Punk.")
print("• 'Kendrick Lamar' is recommended via Travis Scott.")
print("• 'Drake' is NOT recommended because Drake is already directly connected to The Weeknd!")
print("• Output is in alphabetical order:", weeknd_recs == sorted(weeknd_recs))


Direct relationships for 'The Weeknd':
{'Travis Scott', 'Drake', 'Ariana Grande', 'Daft Punk'}

Recommendations for 'The Weeknd' (Exactly 2 links away, not already directly related):
['Kanye West', 'Kendrick Lamar', 'Pharrell Williams']

Notice:
• 'Pharrell Williams' and 'Kanye West' are recommended via Daft Punk.
• 'Kendrick Lamar' is recommended via Travis Scott.
• 'Drake' is NOT recommended because Drake is already directly connected to The Weeknd!
• Output is in alphabetical order: True


### 🛡️ Automated Comprehensive Test Suite for Milestone 3


In [5]:
def run_milestone3_tests():
    disc = TuneWaveDiscovery()

    # 1. BST insertion and exact find
    disc.addSong("Song C", "C3")
    disc.addSong("Song A", "A1")
    disc.addSong("Song B", "B2")
    disc.addSong("Song D", "D4")

    assert disc.findSong("Song B") == "B2"
    assert disc.findSong("Song Z") is None

    # 2. Prefix search returns alphabetical
    disc.addSong("Sunflower", "SF")
    disc.addSong("Sunroof", "SR")
    disc.addSong("Sunday Morning", "SM")
    disc.addSong("Summer", "SU")

    res = disc.prefixSearch("Sun")
    assert res == ["Sunday Morning", "Sunflower", "Sunroof"], f"Got {res}"

    # 3. Graph relations (undirected)
    disc.addRelation("Artist1", "Artist2")
    assert "Artist2" in disc.artist_network.getDirectRelations("Artist1")
    assert "Artist1" in disc.artist_network.getDirectRelations("Artist2")

    # 4. 2-hop recommendations (A-B-C)
    disc.addRelation("Artist2", "Artist3")
    # Distance from Artist1: Artist2 (1 hop), Artist3 (2 hops)
    recs = disc.recommendArtists("Artist1")
    assert recs == ["Artist3"]

    # 5. Triangle test: add direct link A-C
    disc.addRelation("Artist1", "Artist3")
    # Now Artist3 is 1 hop away, so recommendation must be empty!
    assert disc.recommendArtists("Artist1") == []

    # 6. Alphabetical ordering of multiple 2-hop artists
    disc.addRelation("Artist2", "ArtistZ")
    disc.addRelation("Artist2", "ArtistM")
    disc.addRelation("Artist2", "ArtistA")
    recs2 = disc.recommendArtists("Artist1")
    assert recs2 == ["ArtistA", "ArtistM", "ArtistZ"]

    # 7. Isolated artist
    assert disc.recommendArtists("GhostArtist") == []

    print("🎉 ALL MILESTONE 3 TESTS PASSED WITH 100% SUCCESS!")

run_milestone3_tests()


🎉 ALL MILESTONE 3 TESTS PASSED WITH 100% SUCCESS!


## 📊 Complexity & Performance Summary

| Operation | Data Structure | Time Complexity | Space Complexity |
| :--- | :--- | :---: | :---: |
| `addSong(title, songId)` | Binary Search Tree | **$O(h)$** | **$O(1)$** |
| `findSong(title)` | Binary Search Tree | **$O(h)$** | **$O(1)$** |
| `prefixSearch(prefix)` | Pruned In-Order BST | **$O(h + K)$** | **$O(K)$** |
| `addRelation(artistA, artistB)` | Undirected Adjacency List | **$O(1)$** | **$O(1)$** |
| `recommendArtists(artist)` | 2-Hop BFS Neighborhood | **$O(d_1 \cdot ar{d} + M \log M)$** | **$O(M)$** |

Milestone 3 is complete and verified!
